In [ ]:
import chromadb
import numpy as np

docs = [
    "book flight ticket",
    "cancel booking refund",
    "hotel room reservation",
    "baggage allowance policy",
    "airport security rules",
    "meal options onboard",
]
ids = [f"d{i}" for i in range(len(docs))]


In [ ]:
# Task 1 (guide.md): fill `vocab` buckets used by embed().
vocab = {
    "travel": <replace here>,
    "policy": <replace here>,
    "service": <replace here>,
}

def embed(text: str):
    # Deterministic embedding: count keyword matches into 3 buckets.
    words = text.lower().split()
    vec = [
        sum(w in vocab[k] for w in words)
        for k in ["travel", "policy", "service"]
    ]
    return np.array(vec, dtype=float)

embeddings = [embed(d).tolist() for d in docs]


In [ ]:
client = chromadb.Client()
collection = client.get_or_create_collection(name="simple_index_demo_student")
collection.upsert(ids=ids, documents=docs, embeddings=embeddings)

print("Stored", collection.count(), "vectors in ChromaDB")

# Load them back so the later code uses the DB data.
data = collection.get(include=["documents", "embeddings"])
docs = data["documents"]
embeddings = data["embeddings"]


In [ ]:
# Task 2 (MCQ in guide.md): linear scan checks every stored vector.
def linear_search(query, top_k=2):
    q = embed(query)
    scores = []
    checks = 0

    for i, e in enumerate(embeddings):
        checks += 1
        dist = np.linalg.norm(q - np.array(e))  # Euclidean distance
        scores.append((dist, docs[i]))

    scores.sort(key=lambda x: x[0])
    return scores[:top_k], checks

query = "flight rules"
linear_top, linear_checks = linear_search(query)
print("Query:", query)
print("Top results:", linear_top)
print("Distance checks:", linear_checks)


In [ ]:
# Task 3 (guide.md): build the toy index by putting each vector in one bucket.
index_groups = {0: [], 1: [], 2: []}

for i, e in enumerate(embeddings):
    # Replace this line (Task 3) so group is 0, 1, or 2.
    group = <replace here>
    index_groups[group].append(i)

def indexed_search(query, top_k=2):
    q = embed(query)
    target_group = int(np.argmax(q))
    candidate_ids = index_groups[target_group]

    scores = []
    checks = 0
    for i in candidate_ids:
        checks += 1
        dist = np.linalg.norm(q - np.array(embeddings[i]))
        scores.append((dist, docs[i]))

    scores.sort(key=lambda x: x[0])
    return scores[:top_k], checks, target_group

indexed_top, indexed_checks, used_group = indexed_search(query)
print("Used group:", used_group)
print("Top results:", indexed_top)
print("Distance checks:", indexed_checks)


In [ ]:
# Task 5 (guide.md): saved checks = linear_checks - indexed_checks
print("\nComparison")
print("- Linear scan checks:", linear_checks)
print("- Indexed search checks:", indexed_checks)

saved_checks = <replace here>
print("- Saved checks:", saved_checks)
